In [0]:
use catalog identifier(:catalog)

In [0]:

-- 1. Null or duplicate order keys
SELECT CASE WHEN COUNT(*) > 0 THEN raise_error('Critical DQ error: null order key') ELSE 0 END AS check_null_orderkey FROM gold.fact_sales WHERE l_orderkey IS NULL;
SELECT CASE WHEN COUNT(*) > 0 THEN raise_error('Critical DQ error: duplicate order line') ELSE 0 END AS check_duplicate_orderkey FROM (SELECT l_orderkey, l_linenumber FROM gold.fact_sales GROUP BY l_orderkey, l_linenumber HAVING COUNT(*) > 1);

-- 2. Null or duplicate supplier keys
SELECT CASE WHEN COUNT(*) > 0 THEN raise_error('Critical DQ error: null supplier key') ELSE 0 END AS check_null_suppkey FROM gold.dim_supplier WHERE s_suppkey IS NULL;
SELECT CASE WHEN COUNT(*) > 0 THEN raise_error('Critical DQ error: duplicate supplier key') ELSE 0 END AS check_duplicate_suppkey FROM (SELECT s_suppkey FROM gold.dim_supplier GROUP BY s_suppkey HAVING COUNT(*) > 1);

-- 3. Reconciliation: Gold vs Silver
-- If row counts mismatch
SELECT CASE WHEN (SELECT COUNT(*) FROM gold.fact_sales) <> (SELECT COUNT(*) FROM DDEV.silver.lineitem) THEN raise_error('Critical DQ error: Gold and Silver row count mismatch') ELSE 0 END AS check_row_count;
-- If revenues mismatch
SELECT CASE WHEN COALESCE((SELECT SUM(net_revenue) FROM gold.fact_sales),0) <> COALESCE((SELECT SUM(l_extendedprice * (1 - l_discount)) FROM DDEV.silver.lineitem),0) THEN raise_error('Critical DQ error: Gold and Silver revenue mismatch') ELSE 0 END AS check_revenue;
